In [2]:
import pandas as pd

In [3]:
# First things first: Kuziemko exclusions
exclude_countries = [
  'Australia',
  'Belgium',
  'Canada',
  'Czechoslovakia',
  'Denmark',
  'Germany',
  'Ireland',
  'Italy',
  'Japan',
  'Luxembourg',
  'Netherlands',
  'New Zealand',
  'Norway',
  'Singapore',
  'Spain',
  'Sweden',
  'West Germany',
  'Yugoslavia',
]

In [4]:
aid_df = pd.read_csv('aid_panel.csv')

In [5]:
aid_df = aid_df[~aid_df['country'].isin(exclude_countries)]

In [6]:
aid_df['year'].min(), aid_df['year'].max()

(1946, 2020)

In [7]:
member_df = pd.read_csv('elected_members_2020.csv')
member_df = member_df.drop(columns='id')
member_df = member_df[~member_df['country'].isin(exclude_countries)]

In [8]:
member_df['sc_member'] = True

In [9]:
member_df

,year,country,sc_member
1,1946,Brazil,True
2,1946,Egypt,True
3,1946,Mexico,True
5,1946,Poland,True
8,1947,Brazil,True
...,...,...,...
665,2020,Niger,True
666,2020,Saint Vincent and the Grenadines,True
667,2020,South Africa,True
668,2020,Tunisia,True


In [10]:
member_df[member_df['country'].str.contains('United')]

,year,country,sc_member
328,1986,United Arab Emirates,True
337,1987,United Arab Emirates,True


## Merge

In [11]:
aid_sc_df = aid_df.merge(
  member_df,
  how='left',
  on=['country','year'],
  indicator=True,
)
aid_sc_df['_merge'].value_counts()

_merge
left_only     8147
both           480
right_only       0
Name: count, dtype: int64

In [12]:
sc_aid_df = member_df.merge(
  aid_df,
  how='left',
  on=['country','year'],
  indicator=True,
)
sc_aid_df['_merge'].value_counts()

_merge
both          480
left_only      39
right_only      0
Name: count, dtype: int64

In [13]:
# sc_aid_df[sc_aid_df['_merge'] == 'left_only'].iloc[:20]
# sc_aid_df[sc_aid_df['_merge'] == 'left_only'].iloc[20:30]
# sc_aid_df[sc_aid_df['_merge'] == 'left_only'].iloc[30:40]
with pd.option_context('display.max_rows', None):
  display(sc_aid_df[sc_aid_df['_merge'] == 'left_only'])

,year,country,sc_member,total_const_m,_merge
7,1947,Syria,True,NaN,left_only
10,1948,Syria,True,NaN,left_only
11,1948,Ukraine,True,NaN,left_only
12,1949,Argentina,True,NaN,left_only
13,1949,Cuba,True,NaN,left_only
14,1949,Egypt,True,NaN,left_only
15,1949,Ukraine,True,NaN,left_only
16,1950,Cuba,True,NaN,left_only
18,1950,Egypt,True,NaN,left_only
68,1962,Romania,True,NaN,left_only


Notes:

* Syria 1947, 1948 == 0 ✅ Syria starts in 1952
* **Ukraine 1948, 1949** should stay NaN! (Was part of USSR but had its own seat...)
* Argentina 1949 == 0 ✅
* Cuba 1949, 1950 == 0 ✅
* Egypt 1950 == 0 ✅
* Yugoslavia 1950 == 0 ✅
* New Zealand 1954 == 0 ✅
* New Zealand 1955 == 0 ✅
* Sweden 1957 == 0 ✅
* Sweden 1958 == 0 ✅
* Ireland 1962 == 0 ✅
* Romania 1962 == 0 ✅
* Czechoslovakia 1964 == 0 ✅
* Bulgaria 1966, 1967 == 0 ✅
* New Zealand 1966 == 0 ✅
* Algeria 1968 == 0 ✅
* Denmark 1968 == 0 ✅
* Hungary 1968, 1969 == 0 ✅
* Zambia 1969 == 0 ✅
* Japan 1971 == 0 ✅
* Poland 1971 == 0 ✅
* Somalia 1972 == 0 ✅
* Yugoslavia 1973 == 0 ✅
* **Belarus 1974, 1975** should stay NaN! (Same as Ukraine)
* Iraq 1974, 1975 == 0 ✅ (1974 onwards)
* Japan 1975 == 0 ✅
* Sweden 1975 == 0 ✅
* Libya == 0 ✅ (1972 to 2004)
* Kuwait 1978, 1979 == 0 ✅ (none until 1991)
* Nigeria 1978, 1979 == 0 ✅ (none 1977-1985 inclusive)
* East Germany 1980, 1981 == 0 ✅
* Ireland 1981, 1982 == 0 ✅
* Malta 1983, 1984 == 0 ✅ (none 1979-1988 inclusive)
* **Ukraine 1984, 1985** should stay NA
* Bulgaria 1986, 1987 == 0 ✅ (none until 1990)
* United Arab Emirates 1986, 1987 == 0 ✅ (none until 2002)
* South Korea 1997 == 0 ✅ (weird. seems legit though)
* Ireland 2002... should really stay NaN. Sudden dip between 2001 and 2003?

Ok! Now, the goal is, to predict increase in aid from year-before-1st-year to 1st-year on UNSC.

So, the "unit of observation" here is literally just a 2-year "spell" on the UNSC. So, from above, that's `sc_aid_df`.

In [14]:
# Of the 40 NaN countries after merge, this specifies which ones should actually
# stay as NaN
na_list = [
  'Ukraine_1948',
  'Ukraine_1949',
  'Belarus_1974',
  'Belarus_1975',
  'Ukraine_1984',
  'Ukraine_1985',
]
one_year_terms = [
  'Mexico_1946',
  'Netherlands_1946',
  'Egypt_1946',
  'Yugoslavia_1956',
  'Philippines_1957',
  'Poland_1960',
  'Türkiye_1961',
  'Liberia_1961',
  'Ireland_1962',
  'Romania_1962',
  'Philippines_1963',
  'Malaysia_1965',
  'Uganda_1966',
  # Bc no 2021 data
  'Niger_2020',
  'Viet Nam_2020',
  'Saint Vincent and the Grenadines_2020',
  'Estonia_2020',
  'Tunisia_2020',
]
sc_aid_df['cyear'] = sc_aid_df['country'] + '_' + sc_aid_df['year'].apply(lambda x: str(x))
sc_aid_df = sc_aid_df[~sc_aid_df['cyear'].isin(one_year_terms)]

In [15]:
sc_aid_df['term_year'] = 1
sc_aid_df

,year,country,sc_member,total_const_m,_merge,cyear,term_year
0,1946,Brazil,True,187.363835,both,Brazil_1946,1
3,1946,Poland,True,2449.891067,both,Poland_1946,1
4,1947,Brazil,True,12.770138,both,Brazil_1947,1
5,1947,Colombia,True,10.805501,both,Colombia_1947,1
6,1947,Poland,True,1737.721022,both,Poland_1947,1
...,...,...,...,...,...,...,...
509,2019,Poland,True,15.062183,both,Poland_2019,1
510,2019,South Africa,True,631.945424,both,South Africa_2019,1
511,2020,Dominican Republic,True,74.615644,both,Dominican Republic_2020,1
513,2020,Indonesia,True,177.840499,both,Indonesia_2020,1


In [16]:
sc_aid_df['term_count'] = sc_aid_df.groupby(['country'])['term_year'].cumsum()
sc_aid_df.head()

,year,country,sc_member,total_const_m,_merge,cyear,term_year,term_count
0,1946,Brazil,True,187.363835,both,Brazil_1946,1,1
3,1946,Poland,True,2449.891067,both,Poland_1946,1,1
4,1947,Brazil,True,12.770138,both,Brazil_1947,1,2
5,1947,Colombia,True,10.805501,both,Colombia_1947,1,1
6,1947,Poland,True,1737.721022,both,Poland_1947,1,2


In [17]:
sc_aid_df['term_year'] = (sc_aid_df['term_count'] + 1) % 2

In [18]:
sc_aid_df['total_const_m'] = sc_aid_df['total_const_m'].apply(lambda x: 0.0 if pd.isna(x) else x)

In [19]:
sc_aid_df = sc_aid_df.rename(columns={'_merge': 'greenbook'})

In [20]:
sc_aid_df['greenbook'].value_counts()

greenbook
both          466
left_only      38
right_only      0
Name: count, dtype: int64

In [21]:
sc_aid_df['greenbook'] = sc_aid_df['greenbook'].apply(lambda x: True if x == 'both' else False)

In [22]:
# sc_aid_df['term_year'] = sc_aid_df['term_year'] + 1
# sc_aid_df['term_count'] = sc_aid_df['term_count'] + 1

In [23]:
sc_aid_df

,year,country,sc_member,total_const_m,greenbook,cyear,term_year,term_count
0,1946,Brazil,True,187.363835,True,Brazil_1946,0,1
3,1946,Poland,True,2449.891067,True,Poland_1946,0,1
4,1947,Brazil,True,12.770138,True,Brazil_1947,1,2
5,1947,Colombia,True,10.805501,True,Colombia_1947,0,1
6,1947,Poland,True,1737.721022,True,Poland_1947,1,2
...,...,...,...,...,...,...,...,...
509,2019,Poland,True,15.062183,True,Poland_2019,1,10
510,2019,South Africa,True,631.945424,True,South Africa_2019,0,5
511,2020,Dominican Republic,True,74.615644,True,Dominican Republic_2020,1,2
513,2020,Indonesia,True,177.840499,True,Indonesia_2020,1,8


In [24]:
sc_aid_df['term_year'].value_counts()

term_year
0    252
1    252
Name: count, dtype: int64

Ok, once again, exceptions:

* Mexico only on it for **one year** in 1946
* Egypt only on it for **one year** in 1946
* Netherlands only on it for **one year** in 1946

In [25]:
cyear_df = sc_aid_df.sort_values(by=['country','year']).copy()
cyear_df

,year,country,sc_member,total_const_m,greenbook,cyear,term_year,term_count
99,1968,Algeria,True,99.880000,True,Algeria_1968,0,1
107,1969,Algeria,True,11.280316,True,Algeria_1969,1,2
256,1988,Algeria,True,0.156886,True,Algeria_1988,0,3
262,1989,Algeria,True,0.191735,True,Algeria_1989,1,4
381,2004,Algeria,True,5.574756,True,Algeria_2004,0,5
...,...,...,...,...,...,...,...,...
261,1988,Zambia,True,45.167538,True,Zambia_1988,1,6
223,1983,Zimbabwe,True,143.364653,True,Zimbabwe_1983,0,1
232,1984,Zimbabwe,True,103.390929,True,Zimbabwe_1984,1,2
287,1991,Zimbabwe,True,79.462459,True,Zimbabwe_1991,0,3


In [26]:
expected = 0
for ridx, row in cyear_df.iterrows():
  if row['term_year'] == expected:
    expected = (expected + 1) % 2
  else:
    print(row)

Perfect. Now we have all 2-year spells. Collapse each pair of rows into one row

In [27]:
cyear_df = cyear_df.reset_index(drop=True)

In [29]:
spell_df = cyear_df.groupby(cyear_df.index // 2).agg(
  country=('country','first'),
  y0=('year','first'),
  y1=('year','last'),
  aid0=('total_const_m','first'),
  aid1=('total_const_m','last'),
)
spell_df['aid_diff'] = spell_df['aid1'] - spell_df['aid0']
spell_df

,country,y0,y1,aid0,aid1,aid_diff
0,Algeria,1968,1969,99.880000,11.280316,-88.599684
1,Algeria,1988,1989,0.156886,0.191735,0.034849
2,Algeria,2004,2005,5.574756,3.271398,-2.303358
3,Angola,2003,2004,227.583961,161.007484,-66.576477
4,Angola,2015,2016,63.266901,72.196048,8.929147
...,...,...,...,...,...,...
247,Zambia,1969,1970,0.000000,1.076593,1.076593
248,Zambia,1979,1980,99.341085,127.298409,27.957324
249,Zambia,1987,1988,45.440911,45.167538,-0.273373
250,Zimbabwe,1983,1984,143.364653,103.390929,-39.973724


In [30]:
spell_df.to_csv('unsc_spells.csv', index=False)